# 🔀 Python Sorting — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> Sorting is like organising a hand of playing cards.
> You can pick the smallest card each time (selection — slow),
> split the deck in two and merge the halves (merge sort — fast and stable),
> or pick a card as a pivot and put everything smaller on its left (quicksort — fast in practice).
> Different strategies cost differently. Python uses Timsort — the hybrid.

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 1 | [What Is Sorting? The Visual Model](#1) |
| 2 | [Built-in Sort — All Forms](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Merge Sort — Divide and Conquer](#5) |
| 6 | [Pattern 2: Quick Sort — Lomuto Partition](#6) |
| 7 | [Pattern 3: Counting / Bucket Sort — O(n)](#7) |
| 8 | [Pattern 4: Custom Comparator — sort by key](#8) |
| 9 | [Pattern 5: Built-in Mastery — sort() vs sorted()](#9) |
| 10 | [The Sorting Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>

## 1. What Is Sorting? The Visual Model

```
MERGE SORT — divide then conquer

  [5, 3, 8, 1, 2]
       /       \
  [5, 3, 8]  [1, 2]
    /    \      / \
  [5, 3] [8]  [1] [2]
  /    \   |    |   |
 [5]  [3] [8]  [1] [2]
  \   /    |    \  /
  [3, 5]  [8]  [1, 2]
      \   /      |
   [3, 5, 8]  [1, 2]
         \      /
       [1, 2, 3, 5, 8]     ← merge two sorted halves

QUICKSORT — partition around pivot

  [3, 6, 8, 10, 1, 2, 1]   pivot = last element = 1
  i = -1  (boundary: elements <= pivot go here)

  j=0: 3 > 1, skip
  j=1: 6 > 1, skip
  j=2: 8 > 1, skip
  j=3: 10> 1, skip
  j=4: 1 <=1, i=0, swap(0,4) → [1, 6, 8, 10, 3, 2, 1]
  j=5: 2 > 1, skip
  place pivot: swap(i+1, end) → [1, 1, 8, 10, 3, 2, 6]
  pivot at index 1. Left=[1], Right=[8,10,3,2,6]

COUNTING SORT — no comparisons

  input = [4, 2, 2, 8, 3, 3, 1]
  count = [0, 1, 2, 2, 1, 0, 0, 0, 1]   # index = value
          [0  1  2  3  4  5  6  7  8]
  output = [1, 2, 2, 3, 3, 4, 8]         # write each value count times

WHY MERGE SORT IS O(n log n):
  log n levels of splitting × O(n) merge work per level = O(n log n)
  It's the theoretical lower bound for comparison-based sorting.
```

<a id='2'></a>

## 2. Built-in Sort — All Forms

In [ ]:
# Python's built-in sort — all the ways to call it

nums = [5, 3, 8, 1, 2]

# sort() — modifies in place, returns None
nums.sort()
print(f"sort() in-place ascending:  {nums}")

nums.sort(reverse=True)
print(f"sort() in-place descending: {nums}")

# sorted() — returns new list, original unchanged
original = [5, 3, 8, 1, 2]
new_list = sorted(original)
print(f"sorted() new list:   {new_list}")
print(f"original unchanged:  {original}")

# key= parameter — sort by anything
words = ["banana", "fig", "apple", "cherry"]
by_len = sorted(words, key=len)
print(f"sorted by length:    {by_len}")

# sort tuples by second element
pairs = [(1, 3), (2, 1), (3, 2)]
by_second = sorted(pairs, key=lambda x: x[1])
print(f"sorted by 2nd elem:  {by_second}")

# stability: equal keys keep original order
data = [(1, 'b'), (2, 'a'), (1, 'a'), (2, 'b')]
stable = sorted(data, key=lambda x: x[0])
print(f"stable sort:         {stable}")
print("Built-in sort forms demonstrated.")

<a id='3'></a>

## 3. The Core API — All Operations

```
OPERATION                          COMPLEXITY   WHAT IT DOES
────────────────────────────────────────────────────────────────────
list.sort()                        O(n log n)   in-place Timsort
sorted(iterable)                   O(n log n)   returns new sorted list
list.sort(key=fn)                  O(n log n)   sort by extracted key
list.sort(reverse=True)            O(n log n)   descending
sorted(d, key=d.get)               O(n log n)   sort dict by values
heapq.nsmallest(k, iterable)       O(n log k)   k smallest elements
heapq.nlargest(k, iterable)        O(n log k)   k largest elements
counting_sort(arr, k)              O(n + k)     non-comparison sort
functools.cmp_to_key(cmp_fn)       O(1)         wrap comparator for key=

THINGS YOU DO NOT DO:
❌  Write bubble sort for interviews — use Python built-in
❌  Use cmp_to_key when a simple key= lambda works
❌  Forget list.sort() returns None (not the sorted list)
❌  Use counting sort when values are negative or unbounded
❌  Assume sorted() works on dicts without key= (sorts by key by default)
```

In [ ]:
import functools

# Demo 1: sort dict by value
freq = {'a': 3, 'b': 1, 'c': 2}
by_val = sorted(freq, key=freq.get, reverse=True)
print(f"dict keys by value desc: {by_val}")

# Demo 2: sort by multiple keys (primary then secondary)
students = [('Alice', 90), ('Bob', 85), ('Carol', 90), ('Dave', 85)]
multi = sorted(students, key=lambda s: (-s[1], s[0]))
print(f"by score desc, name asc: {multi}")

# Demo 3: cmp_to_key for custom comparison logic
# Example: sort numbers as strings for largest number formation (LC 179)
def compare(a, b):
    # which ordering 'ab' or 'ba' gives the larger number?
    if str(a) + str(b) > str(b) + str(a):
        return -1   # a should come first
    else:
        return 1

nums = [3, 30, 34, 5, 9]
nums.sort(key=functools.cmp_to_key(compare))
print(f"largest number order: {nums}")
print(f"  → concatenated: {''.join(map(str, nums))}")

# Demo 4: sort() returns None — common mistake
arr = [3, 1, 2]
result = arr.sort()   # returns None!
print(f"sort() return value: {result}")
print(f"arr itself is sorted: {arr}")
print("Core API demo done.")

<a id='4'></a>

## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                    ALGORITHM
────────────────────────────────────────────────────────────────
Just need sorted output                  Python built-in sort()
Sort by custom rule / multiple keys      sort(key=lambda ...)
Comparator logic (a vs b ordering)       cmp_to_key + sort
Values in small range [0..k]             Counting sort O(n+k)
Need stable sort (guaranteed)            Timsort is stable ✓
Implement from scratch (interview)       Merge sort (stable, O(n log n))
In-place with O(1) extra space           Quicksort (avg O(n log n))
k smallest/largest (don't need sorted)   Heap — O(n log k)
```

<a id='5'></a>

## 5. 🧩 Pattern 1: Merge Sort — Divide and Conquer

---

```
PROBLEM:
  Sort an array. Implement merge sort for interview from scratch.

TRICK:
  Split in half until single elements (base case).
  Merge two sorted halves by comparing front elements — O(n) per level.
  log n levels → O(n log n) total. Stable.

SLOW MOTION TRACE on [5,3,8,1,2]:
  level 0: [5,3,8,1,2]
  level 1: [5,3,8]        [1,2]
  level 2: [5,3] [8]      [1] [2]
  level 3: [5][3] [8]     [1] [2]
  merge up:
    [3,5]  [8]       [1,2]
    [3,5,8]          [1,2]
    [1,2,3,5,8]

  Merge [3,5] and [8]:
    compare 3,8 → take 3 → [3]
    compare 5,8 → take 5 → [3,5]
    nothing left in left → append rest [8] → [3,5,8]

KEY INSIGHT:
  Merging two sorted arrays is O(n). Splitting creates log n levels.
  Merge sort is the stable O(n log n) workhorse — use for interviews.

TIME:  O(n log n) — log n levels × O(n) merge per level
SPACE: O(n)       — temporary arrays during merge
```

In [ ]:
from typing import List

def merge_sort(arr: List[int]) -> List[int]:
    """
    Merge Sort — top-down divide and conquer.
    Approach: recursively split, merge sorted halves.
    Args:
        arr (List[int]): input array.
    Returns:
        List[int]: new sorted array.
    Time:  O(n log n) — log n levels, O(n) merge at each level
    Space: O(n)       — temporary left/right arrays at each merge
    """
    if len(arr) <= 1:                   # base case — single element is sorted
        return arr

    mid = len(arr) // 2
    left = merge_sort(arr[:mid])        # sort left half
    right = merge_sort(arr[mid:])       # sort right half
    return _merge(left, right)          # combine two sorted halves

def _merge(left: List[int], right: List[int]) -> List[int]:
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:         # take smaller front element
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])             # append remaining (already sorted)
    result.extend(right[j:])
    return result

# Slow motion on [5,3,8,1,2]:
# split: [5,3,8] [1,2] → [5,3][8] [1][2]
# merge [5][3]=[3,5], [3,5]+[8]=[3,5,8], [1]+[2]=[1,2]
# final merge [3,5,8]+[1,2]=[1,2,3,5,8]

def test_harness(fn):
    tests = [
        ([5,3,8,1,2],      [1,2,3,5,8]),
        ([1],              [1]),
        ([],               []),
        ([2,1],            [1,2]),
        ([3,3,3],          [3,3,3]),
        ([-3,0,5,-1,2],    [-3,-1,0,2,5]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(merge_sort)
print("merge_sort defined.")

<a id='6'></a>

## 6. 🧩 Pattern 2: Quick Sort — Lomuto Partition

---

```
PROBLEM:
  Sort an array in-place. Implement quick sort.

TRICK:
  Pick last element as pivot. Scan array: elements <= pivot go left.
  After scan, place pivot at boundary+1. Recurse on left and right sub-arrays.

SLOW MOTION TRACE on [3,6,8,10,1,2,1]  pivot=1 (last):
  i = lo-1 = -1     (boundary: elements ≤ pivot)
  j=0: arr[j]=3 > 1, skip
  j=1: arr[j]=6 > 1, skip
  j=2: arr[j]=8 > 1, skip
  j=3: arr[j]=10> 1, skip
  j=4: arr[j]=1 ≤ 1, i=0, swap(0,4) → [1,6,8,10,3,2,1]
  j=5: arr[j]=2 > 1, skip
  place pivot at i+1=1: swap(1,6) → [1,1,8,10,3,2,6]
  pivot idx=1. recurse left=[1], right=[8,10,3,2,6]

KEY INSIGHT:
  After partition, pivot is in its final position — never moves again.
  Average O(n log n). Worst case O(n²) on sorted input (random pivot fixes).

TIME:  O(n log n) avg, O(n²) worst (sorted input, last-element pivot)
SPACE: O(log n)   — recursion stack depth
```

In [ ]:
import random

def quick_sort(arr: List[int], lo: int = 0, hi: int = None) -> None:
    """
    Quick Sort — Lomuto partition, in-place.
    Approach: partition around pivot (last element), recurse both sides.
    Args:
        arr (List[int]): array to sort, modified in place.
        lo (int): left boundary (inclusive).
        hi (int): right boundary (inclusive).
    Returns:
        None — modifies arr in place.
    Time:  O(n log n) avg — O(n²) worst case on already-sorted input
    Space: O(log n)  — recursion stack depth
    """
    if hi is None:
        hi = len(arr) - 1

    if lo >= hi:                          # base case — 0 or 1 element
        return

    pivot_idx = _partition(arr, lo, hi)
    quick_sort(arr, lo, pivot_idx - 1)   # sort left of pivot
    quick_sort(arr, pivot_idx + 1, hi)   # sort right of pivot

def _partition(arr: List[int], lo: int, hi: int) -> int:
    # randomize pivot to avoid O(n²) on sorted input
    rand_idx = random.randint(lo, hi)
    arr[rand_idx], arr[hi] = arr[hi], arr[rand_idx]

    pivot = arr[hi]                       # pivot is now at the end
    i = lo - 1                            # boundary: arr[lo..i] <= pivot

    for j in range(lo, hi):
        if arr[j] <= pivot:               # element belongs in left zone
            i += 1
            arr[i], arr[j] = arr[j], arr[i]

    arr[i + 1], arr[hi] = arr[hi], arr[i + 1]  # pivot to its final position
    return i + 1

# Slow motion on [3,6,8,10,1,2,1] with pivot=1 (last):
# i=-1, j scans: only j=4(val=1) and j=5(val... wait val=2>1), only j=4 qualifies
# after loop: i=0, swap(i+1=1, hi=6) → pivot at index 1

def test_harness(fn):
    import copy
    tests = [
        ([5,3,8,1,2],   [1,2,3,5,8]),
        ([1],           [1]),
        ([],            []),
        ([2,1],         [1,2]),
        ([3,3,3],       [3,3,3]),
        ([-3,0,5,-1,2], [-3,-1,0,2,5]),
    ]
    passed = 0
    for *inputs, expected in tests:
        arr = copy.deepcopy(inputs[0])
        fn(arr)
        status = "PASSED" if arr == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={arr}")
        passed += (arr == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(quick_sort)
print("quick_sort defined.")

<a id='7'></a>

## 7. 🧩 Pattern 3: Counting / Bucket Sort — O(n+k)

---

```
PROBLEM:
  Sort when values are integers in a known bounded range [0..k].

TRICK:
  Count occurrences. Output each value its count times. No comparisons.
  O(n+k). For top-K frequency problems, use bucket sort on frequencies.

SLOW MOTION TRACE on [4,2,2,8,3,3,1]:
  k=8 (max value)
  count = [0]*9
  after counting: count = [0,1,2,2,1,0,0,0,1]
               idx:        [0 1 2 3 4 5 6 7 8]
  output: write 0 zeros, 1 one, 2 twos, 2 threes, 1 four, 1 eight
  result = [1,2,2,3,3,4,8]

BUCKET SORT for top-K frequency (LC 347):
  bucket[freq] = [list of nums with that frequency]
  bucket[1] = [5, 3]
  bucket[2] = [1]
  Read from bucket[n] down to bucket[0], take first k elements.

KEY INSIGHT:
  Breaking the O(n log n) comparison barrier requires domain knowledge
  about the input — bounded integers or frequencies.

TIME:  O(n + k) — n for counting, k for output
SPACE: O(n + k) — count array of size k+1
```

In [ ]:
from collections import Counter

def counting_sort(arr: List[int]) -> List[int]:
    """
    Counting Sort — O(n+k) non-comparison sort for bounded integers.
    Args:
        arr (List[int]): non-negative integers in range [0..max(arr)].
    Returns:
        List[int]: new sorted array.
    Time:  O(n + k) — n to count, k to output
    Space: O(n + k) — count array size k+1, output size n
    """
    if not arr:
        return []

    k = max(arr)                          # range of values
    count = [0] * (k + 1)                # count[v] = how many times v appears

    for val in arr:
        count[val] += 1                   # tally each value

    result = []
    for val, freq in enumerate(count):
        result.extend([val] * freq)       # output val freq times

    return result

def top_k_frequent_bucket(nums: List[int], k: int) -> List[int]:
    """
    LC 347 — Top K Frequent Elements (bucket sort approach).
    Approach: bucket by frequency, read from high-freq buckets first.
    Time:  O(n) — count O(n) + bucket O(n) + read O(n)
    Space: O(n) — freq map + buckets
    """
    freq = Counter(nums)                  # value → frequency
    buckets = [[] for _ in range(len(nums) + 1)]  # index = frequency

    for val, f in freq.items():
        buckets[f].append(val)            # put val in its frequency bucket

    result = []
    for f in range(len(buckets) - 1, 0, -1):  # high freq first
        for val in buckets[f]:
            result.append(val)
            if len(result) == k:
                return result

    return result

# Slow motion on [4,2,2,8,3,3,1]:
# count=[0,1,2,2,1,0,0,0,1], output 1 one, 2 twos, 2 threes, 1 four, 1 eight
# → [1,2,2,3,3,4,8]

def test_harness(fn):
    tests = [
        ([4,2,2,8,3,3,1], [1,2,2,3,3,4,8]),
        ([1],             [1]),
        ([],              []),
        ([3,3,3],         [3,3,3]),
        ([0,1,0,2,1],     [0,0,1,1,2]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(counting_sort)
print("counting_sort and top_k_frequent_bucket defined.")

<a id='8'></a>

## 8. 🧩 Pattern 4: Custom Comparator — sort by key

---

```
PROBLEM:
  Sort intervals by start (LC 56), or form largest number (LC 179),
  or sort people by height then name.

TRICK 1 — key= lambda:
  Sort intervals by end time: sorted(intervals, key=lambda x: x[1])
  Sort strings by length then alpha: sorted(words, key=lambda w: (len(w), w))

TRICK 2 — cmp_to_key for LC 179 Largest Number:
  Compare ab vs ba as strings. If '34' + '3' = '343' > '3' + '34' = '334',
  then 34 should come before 3.

SLOW MOTION TRACE on LC 179 [3, 30, 34, 5, 9]:
  compare(3, 30): '330' vs '303' → '330' > '303' → 3 before 30
  compare(34, 3): '343' vs '334' → '343' > '334' → 34 before 3
  compare(5, 34): '534' vs '345' → '534' > '345' → 5 before 34
  compare(9, 5):  '95' vs '59'   → '95'  > '59'  → 9 before 5
  sorted: [9, 5, 34, 3, 30]
  joined: '9534330'

KEY INSIGHT:
  Python sort is stable and takes any key function. cmp_to_key wraps
  a comparator for cases where relative ordering depends on both elements.

TIME:  O(n log n) — standard sort with O(1) or O(k) key extraction
SPACE: O(n)       — key array built once by Timsort
```

In [ ]:
import functools

def largest_number(nums: List[int]) -> str:
    """
    LC 179 — Largest Number
    Approach: custom comparator — ab > ba means a should come first.
    Args:
        nums (List[int]): non-negative integers.
    Returns:
        str: digits of largest number formable by concatenation.
    Time:  O(n log n) — sort with O(k) comparator (k = string length)
    Space: O(n)       — string conversion
    """
    def compare(a: str, b: str) -> int:
        # which order gives the larger concatenation?
        if a + b > b + a:
            return -1   # a should come first (sort ascending by comparison)
        elif a + b < b + a:
            return 1
        return 0

    strs = list(map(str, nums))
    strs.sort(key=functools.cmp_to_key(compare))

    if strs[0] == '0':               # all zeros edge case → '000...' = '0'
        return '0'

    return ''.join(strs)

# Slow motion on [3,30,34,5,9]:
# '3'+'30'='330' vs '30'+'3'='303' → '330'>'303' → 3 before 30
# final sort: ['9','5','34','3','30'] → '9534330'

def sort_intervals_by_start(intervals: List[List[int]]) -> List[List[int]]:
    """
    Sort intervals by start time (prerequisite for LC 56 Merge Intervals).
    Time:  O(n log n)
    Space: O(n)
    """
    return sorted(intervals, key=lambda x: x[0])  # sort by first element

def test_harness(fn):
    tests = [
        ([3, 30, 34, 5, 9],  "9534330"),
        ([10, 2],            "210"),
        ([1],                "1"),
        ([0, 0],             "0"),
        ([0, 1],             "10"),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(largest_number)
print("largest_number and sort_intervals_by_start defined.")

<a id='9'></a>

## 9. 🧩 Pattern 5: Built-in Sort Mastery — sort() vs sorted()

---

```
PROBLEM:
  Know exactly when to use sort() vs sorted(), stability, and key tricks.

TRICK — which to use:
  sort()   → in-place, returns None, modifies original
  sorted() → returns new list, original unchanged, works on any iterable

STABILITY:
  Python's Timsort is stable — equal elements keep their original order.
  Use this when sorting by primary then secondary key:
    sort by name first, then by score → stable sort keeps name order within
    same score groups.

SORT DICT BY VALUE:
  sorted(d.items(), key=lambda kv: kv[1])
  sorted(d, key=d.get, reverse=True)

KEY INSIGHT:
  Timsort exploits existing runs in real-world data — often faster than
  O(n log n) in practice. It's merge sort + insertion sort hybrid.

TIME:  O(n log n) worst, O(n) best (already sorted input)
SPACE: O(n)       — Timsort uses O(n) extra for merge
```

In [ ]:
# sort() vs sorted() — complete mastery

# 1. sort() modifies in place — returns None
arr = [3, 1, 4, 1, 5, 9, 2, 6]
arr.sort()
print(f"sort() result:          {arr}")

# 2. sorted() returns new list
original = [3, 1, 4, 1, 5, 9, 2, 6]
new = sorted(original)
print(f"sorted() new list:      {new}")
print(f"original unchanged:     {original}")

# 3. Stability demo — sort by score, then by name (two-pass stable)
students = [('Charlie', 85), ('Alice', 90), ('Bob', 85), ('Diana', 90)]
students.sort(key=lambda s: s[0])        # first sort by name (stable base)
students.sort(key=lambda s: -s[1])       # then sort by score desc (stable)
print(f"stable two-key sort:    {students}")

# 4. Sort dict by value descending
word_freq = {'the': 100, 'a': 200, 'is': 50, 'and': 150}
top_words = sorted(word_freq, key=word_freq.get, reverse=True)
print(f"words by freq desc:     {top_words}")

# 5. sort() on any iterable with sorted()
gen = (x**2 for x in [3, 1, 4, 1, 5])
sorted_gen = sorted(gen)                  # sorted() accepts any iterable
print(f"sorted generator:       {sorted_gen}")

def test_harness(fn):
    # test stable sort behavior
    tests = [
        ([3,1,2], [1,2,3]),
        ([],      []),
        ([1],     [1]),
        ([-2,0,1,-1,2], [-2,-1,0,1,2]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(sorted)    # sorted() itself as the function under test
print("Built-in sort mastery demonstrated.")

<a id='10'></a>

## 10. The Sorting Decision Map

```
QUESTION TYPE                        KEY ALGORITHM         COMPLEXITY
─────────────────────────────────────────────────────────────────────
Just sort the array                  Python sort()         O(n log n)
Sort by custom key                   sorted(key=lambda)    O(n log n)
Sort where a vs b order matters      cmp_to_key            O(n log n)
Values in range [0..k]               Counting sort         O(n + k)
Sort by frequency (top-k)            Bucket sort           O(n)
Implement from scratch (stable)      Merge sort            O(n log n)
Implement in-place (no extra space)  Quick sort            O(n log n) avg
k smallest/largest (no full sort)    heapq.nsmallest/nlargest  O(n log k)

STABILITY MATTERS WHEN:
  You sort by one key then another, and equal elements in second sort
  must keep the order established by the first sort.
  Python Timsort is always stable — use two separate sorts safely.
```

<a id='11'></a>

## 11. Interview Cheat Sheet

**1. When to reach for each algorithm:**

| Signal | Algorithm |
|--------|----------|
| General sort needed | Python `sort()` or `sorted()` |
| Sort by computed property | `key=lambda` |
| ab vs ba comparison needed | `cmp_to_key` |
| Values 0..k, need O(n) | Counting sort |
| Frequency/bucket grouping | Bucket sort |
| Implement sort in interview | Merge sort |

**2. Core operations — memorize these:**

```python
arr.sort()                                # in-place, returns None
arr.sort(reverse=True)                    # descending
arr.sort(key=lambda x: x[1])             # sort by second element
sorted(arr, key=lambda x: (x[0], -x[1])) # primary asc, secondary desc
sorted(d, key=d.get, reverse=True)        # dict keys by value desc
sorted(arr, key=functools.cmp_to_key(cmp_fn))
```

**3. Common templates:**

```python
# TEMPLATE 1: MERGE SORT
def merge_sort(arr):
    if len(arr) <= 1: return arr
    mid = len(arr) // 2
    L, R = merge_sort(arr[:mid]), merge_sort(arr[mid:])
    res, i, j = [], 0, 0
    while i < len(L) and j < len(R):
        if L[i] <= R[j]: res.append(L[i]); i += 1
        else: res.append(R[j]); j += 1
    return res + L[i:] + R[j:]

# TEMPLATE 2: COUNTING SORT
def counting_sort(arr):
    count = [0] * (max(arr) + 1)
    for v in arr: count[v] += 1
    return [v for v, f in enumerate(count) for _ in range(f)]

# TEMPLATE 3: CUSTOM COMPARATOR (LC 179 style)
import functools
def cmp(a, b): return -1 if a+b > b+a else 1 if a+b < b+a else 0
strs.sort(key=functools.cmp_to_key(cmp))
```

**4. Gotchas:**

```
❌  arr.sort() returns None — don't assign it
❌  Counting sort with negatives — shift values first
❌  Quicksort worst case O(n²) on sorted input — use random pivot
❌  cmp_to_key: return -1 for 'a before b', NOT -1 for 'a is smaller'
✅  Python sort is stable — two-pass sort is safe
✅  For k smallest/largest — heapq is faster than full sort when k << n
```

<a id='12'></a>

## 12. Summary Map

```
SORTING
│
├── Comparison-Based O(n log n)
│     ├── Merge Sort  — stable, O(n) space, interview go-to
│     ├── Quick Sort  — O(log n) space, avg fast, worst O(n²)
│     └── Python Timsort — merge + insertion hybrid, always stable
│
├── Non-Comparison O(n)
│     ├── Counting Sort — values in [0..k]
│     └── Bucket Sort   — group by frequency (top-k pattern)
│
└── Custom Ordering
      ├── key=lambda     — sort by extracted property
      └── cmp_to_key     — sort by pairwise comparison logic

CHOOSE BY:
  general sort needed    → Python sort() — done
  implement from scratch → merge sort (stable, predictable)
  small integer values   → counting / bucket sort for O(n)
  a vs b ordering logic  → cmp_to_key
```

---
*End of Sorting Master Guide — Sean Edition*